> 📄 English: [vectorize_tutorial.ipynb](vectorize_tutorial.ipynb)

# 教程：为你自己的文档做向量搜索准备


In [ ]:
%pip install langchain pypdf langchain-openai --quiet

## 1. 上传你的文档
首先，删除 `/docs` 文件夹中的现有文件，并添加你自己的 PDF 文件。然后运行下面的单元格

In [ ]:
# Create a loader that processes all files in the docs directory
import os
from langchain_community.document_loaders import PyPDFLoader

# Path to docs directory
docs_dir = "./docs"

# Get all files in the directory
all_files = [os.path.join(docs_dir, f) for f in os.listdir(docs_dir) 
             if os.path.isfile(os.path.join(docs_dir, f))]

# Process each file in the directory
documents = []
for file_path in all_files:
    try:
        loader = PyPDFLoader(
            file_path=file_path,
        )
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} chunks from {file_path}")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

print(f"Loaded total of {len(documents)} document chunks")

## 2. 切分文档
将大文档切分成更小的块，以获得更好的 embedding 质量。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into chunks of 1000 characters with 200 characters overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

In [ ]:
# print chunks
for chunk in chunks:
    print(chunk.page_content)
    print("-"*100)

## 3. 生成 embeddings
使用 OpenAI embeddings 将每个块编码为向量。

In [ ]:
from langchain_openai import OpenAIEmbeddings
# define embeddings as default OpenAI embeddings
embeddings = OpenAIEmbeddings()

## 4. 将 embeddings 存储到 Chroma
初始化一个 Chroma 向量存储，并将其持久化到本地。

如果你遇到 "OperationalError: attempt to write a readonly database" 错误——请重启内核并重新运行 notebook。


In [ ]:
from langchain_community.vectorstores import Chroma

# create vector store with Chroma
vectordb = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    persist_directory="db",
    collection_name="my_custom_index"
)
vectordb.persist()

## 5. 示例相似度搜索
在你的向量存储上执行一个相似度搜索查询。

In [ ]:
# Test similarity search
query = "robotics"
results = vectordb.similarity_search(query, k=5)
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}...\n")